# React — Performance

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> **The rule for this whole topic: make it correct → measure → optimise.** Every tool here is
> useless or harmful when applied to a problem you have not measured, and the lessons are
> ordered to make that hard to skip. Nothing in this topic is required to write good React.

## LESSON 68 — Why re-renders happen

Before optimising anything you need to know what actually triggers the work. There are exactly
two answers, and one of them surprises people.

### The two reasons

1. **This component's own state changed.** LESSON 38: *"Updating your component's state
   automatically queues a render."*
2. **Its parent re-rendered.**

The second is the one that matters here, and React states it plainly:

> React normally re-renders a component whenever its parent re-renders.

Note what is *not* on the list. A component does not re-render because its props "changed" —
props changing is a *consequence* of the parent rendering, not a cause. A child with no props at
all re-renders just the same. React does not compare anything before calling your component; it
calls it, then compares the result to decide what to touch in the DOM (LESSON 38's commit step).

So a state change in `App` calls **every** component function in the app. The playground
experiment counts exactly that.

### Which means most "React is slow" is a state-placement problem

If a keystroke re-renders four hundred components, the interesting question is not "how do I
make four hundred renders faster" but "why does a keystroke reach four hundred components".
That is LESSON 66, one topic ago, and it is the fix in a large majority of real cases.

React's own list of principles that make memoization mostly unnecessary opens with two you have
already met:

> 1. When a component wraps other components, let it accept JSX as children…
> 2. Prefer local state and don't lift state up further than necessary.

The first one is worth seeing. A component that renders `{children}` does not re-render those
children when its **own** state changes — the child elements were created by the *parent* and
did not change. The experiment measures this too, and it is the cheapest optimisation in React
because it is just a component boundary.

### Measure first, and measure the right thing

You have the Profiler from LESSON 35. Before changing a line of code:

1. Record the interaction that feels slow.
2. Read **which** components rendered and **how long** the commit took.
3. Decide whether the number is a problem. A commit of 3 ms is not a problem no matter how many
   components it contains.

A render count is not by itself a bug. Rendering is calling a function; hundreds of cheap
function calls are cheap. What costs is a *slow* render — a big list, a heavy calculation — or a
render that happens on every keystroke.

### Key Notes

- Two causes only: own state changed, or the parent re-rendered.
- Props changing is not a cause. React calls the child either way.
- Most slowness is state living too high (LESSON 66) — fix that before reaching for `memo`.
- Measure with the Profiler first. A high render count with a fast commit is not a problem.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/28-rerenders.jsx`, open the console
and press **App state**.

Every component logs its own render, and the tally is also on `window.__renderCounts` so you can
read it without triggering anything. StrictMode double-invokes components in development, so
every count moves in twos — compare components against each other, not against 1.

What one press of **App state** produces (halving StrictMode's doubling):

| component | renders | why |
|---|---|---|
| `App` | 1 | its own state changed |
| `PlainChild` | 1 | its parent re-rendered |
| `Wrapper` | 1 | its parent re-rendered |
| `PassedAsChildren` | 1 | its parent (`App`) created the element |
| `MemoPrimitive` | **0** | LESSON 69 |
| `MemoObject` | 1 | LESSON 69, and the point of it |

Then press **wrapper state**, which changes state *inside* `Wrapper`:

| component | renders |
|---|---|
| `Wrapper` | 1 |
| `PassedAsChildren` | **0** |
| everything else | 0 |

`PassedAsChildren` sits inside `Wrapper` on screen and did not re-render, because `Wrapper` did
not create it — `App` did, and `App` did not run. That is principle 1 above, measured.

Finally press **log the render counts**, which changes no state: nothing renders at all.

### Exercise

**In the playground**, in `28-rerenders.jsx`.

1. Add a `<Sibling />` component next to `PlainChild` that takes **no props at all** and logs its
   render. Press **App state**. Does a component with no props escape the re-render? Say why in
   one sentence, using the two reasons from this lesson.
2. Move `useState` for `count` out of `App` and into a new `<Counter />` component that renders
   only the button. Press it. Which components render now, and how many fewer than before?
3. Wrap the `<ul>` and its four children in a `<Wrapper>` as `children` — that is, move them
   inside the existing `Wrapper` tags. Press **wrapper state** and then **App state**. Explain
   which press re-renders them and which does not, in terms of who created the elements.
4. Open the Profiler (LESSON 35), record one press of **App state**, and write down the commit
   duration in milliseconds. Then answer honestly: is anything in this experiment worth
   optimising?

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** A render-count budget, before you have any tools to fix it.

A page has a header (1 component), a sidebar (12), a table of 200 rows (3 components each) and a
footer (1). Write `l68Renders(where)` returning how many component functions React calls when
state changes in `where` — `"App"`, `"sidebar"` or `"one row"` — assuming state lives at that
level and nothing is memoized.

Then answer in comments: which of the three is the search box's state, if the search box is in
the header and the table shows the results? And what does that tell you about where the fix has
to come from — a `memo` call, or a decision from LESSON 66?

In [ ]:
// Your code here

## LESSON 69 — `memo` and reference equality

`memo` is the tool for the second reason from LESSON 68 — a component re-rendering only because
its parent did.

> `memo` lets you skip re-rendering a component when its props are unchanged.

```jsx
import { memo } from "react";

const Row = memo(function Row({ label }) {
  return <li>{label}</li>;
});
```

`Row` is now a component that React will skip when the props it receives are the same as last
time. Everything else about it is unchanged.

### "Unchanged" means compared by identity, prop by prop

This is the whole lesson. `memo` does a **shallow** comparison: for each prop, `Object.is(old,
new)` — the same comparison you met at LESSON 40 for dependency arrays, and at LESSON 27 for why
you never mutate state.

For a string, a number or a boolean, that does what you expect. For an object, an array or a
function it compares **references**, and a fresh object written in the parent's body is a new
reference on every render:

```jsx
function Parent() {
  const config = { label: "hi" };        // a NEW object, every single render
  const onPing = () => {};               // a NEW function, every single render
  return <MemoChild config={config} onPing={onPing} />;   // memo does nothing
}
```

The child is wrapped in `memo`, the props look identical, and it re-renders every time — because
`{ label: "hi" }` is not `Object.is`-equal to last render's `{ label: "hi" }`. You have paid for
a comparison and gained nothing.

### Two things `memo` does not do

> Even with `memo`, your component will still re-render if its own state changes or if a context
> that it's using changes.

And:

> Memoization is a performance optimization, not a guarantee.

React may re-render a memoized component anyway. Never write code whose *correctness* depends on
a skipped render.

### When it is worth it

React's guidance is unusually direct:

> **Should you add `memo` everywhere?** …If your app is like this site, and most interactions
> are coarse (like replacing a page or a whole section), memoization is usually unnecessary.

It is worth reaching for when a component *re-renders often with the same props* and its
rendering is *expensive*, and there is *perceptible lag*. All three, measured. And if props are
always different, `memo` is useless by construction.

Read that list again against the previous lesson: two of the three conditions are things you
cannot know without the Profiler.

### Key Notes

- `memo` skips a re-render caused by the parent, when every prop is `Object.is`-equal.
- Objects, arrays and functions created during render are new references every time — `memo`
  then does nothing at all.
- It never blocks a re-render from the component's own state or Context.
- Use it when a component re-renders often with the same props, is expensive, and you have felt
  the lag. Otherwise it is cost with no benefit.

### Example

**Runnable — plain JS.** `memo`'s comparison is twelve lines, and writing it removes the mystery
for good. This is the real algorithm, not a model of one.

In [ ]:
// L69 — the comparison memo actually performs

function l69ShallowEqual(previous, next) {
  const a = Object.keys(previous);
  const b = Object.keys(next);
  if (a.length !== b.length) return false;
  return a.every((key) => Object.is(previous[key], next[key]));
}

// --- the four prop shapes from playground experiment 28 ---------------------
const l69Render1 = {
  label: "steady",                                  // a string
  config: { label: "rebuilt every render" },        // an object literal
  onPing: () => {},                                 // a function
};

// the SAME source code, evaluated a second time — a second render
const l69Render2 = {
  label: "steady",
  config: { label: "rebuilt every render" },
  onPing: () => {},
};

console.log("string prop equal? ", Object.is(l69Render1.label, l69Render2.label));
console.log("object prop equal? ", Object.is(l69Render1.config, l69Render2.config));
console.log("function prop equal?", Object.is(l69Render1.onPing, l69Render2.onPing));

console.log("\nwhole props object shallow-equal?", l69ShallowEqual(l69Render1, l69Render2));

// only the string survives the comparison
console.log(
  "just the string prop:",
  l69ShallowEqual({ label: l69Render1.label }, { label: l69Render2.label }),
);

// and the object is not "different" in any way a human would care about
console.log("same contents?", JSON.stringify(l69Render1.config) === JSON.stringify(l69Render2.config));

### Exercise

Part 1 is **runnable — plain JS**; parts 2 and 3 are **in the playground**, in
`28-rerenders.jsx`.

1. Extend `l69ShallowEqual` into `l69Explain(previous, next)` that returns the list of prop names
   that failed the comparison, so a caller can see *why* a memoized component re-rendered. Run it
   on the two render snapshots and print the failing names.
2. In the experiment, `MemoPrimitive` skips its re-render and `MemoObject` does not. Change
   `MemoPrimitive`'s prop from `label="steady"` to `label={`steady ${count}`}` and press **App
   state**. What happens to its render count, and which of React's three conditions for using
   `memo` has now been violated?
3. Give `MemoObject` a second prop that is a plain number, and leave the object prop in place.
   Does adding an equal prop help? Say in one sentence what `memo` would need in order to skip
   this component, and why that is LESSON 70's subject rather than this one's.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** A memo audit, on paper.

Here are five prop objects a parent passes to a memoized child on two consecutive renders. For
each, predict **before running anything** whether `memo` skips the child, then check with
`l69ShallowEqual`:

```js
const l69Cases = [
  ["id + name",        { id: 3, name: "Ada" },              { id: 3, name: "Ada" }],
  ["same array",       { rows: sharedRows },                 { rows: sharedRows }],
  ["new array",        { rows: [1, 2] },                     { rows: [1, 2] }],
  ["handler in state", { onSave: storedHandler },            { onSave: storedHandler }],
  ["extra prop",       { id: 3 },                            { id: 3, tone: undefined }],
];
```

Then answer two questions in comments. The last case has an extra key whose value is `undefined`
— does `memo` skip it, and does that match what you would have guessed from reading the JSX?
And of the five, which two are the shapes you can actually create *without* `useMemo`, just by
where you declare the value?

In [ ]:
// Your code here

## LESSON 70 — `useMemo` and `useCallback`

Two Hooks, one idea, and the same measurement discipline as the last two lessons.

> `useMemo` is a React Hook that lets you cache the result of a calculation between re-renders.

```jsx
const visibleItems = useMemo(() => filterAndSort(items, tab), [items, tab]);
```

React runs the function on the first render and stores the result. On later renders it compares
the dependency array — the same `Object.is`, item by item, as LESSON 40 — and returns the stored
value if nothing changed.

`useCallback` is the same thing for a function:

```jsx
const handleSave = useCallback((draft) => save(id, draft), [id]);
```

`useCallback(fn, deps)` is `useMemo(() => fn, deps)` with nicer syntax. It does not make the
function faster. It keeps the **same function reference** across renders, which is only
interesting because of the previous lesson: a stable reference is what a memoized child needs.

### The two reasons to reach for them

**One: the calculation is genuinely slow.** Then `useMemo` skips work.

**Two: a downstream comparison depends on the identity.** Then `useMemo`/`useCallback` are what
make `memo` — or an Effect's dependency array — actually work. LESSON 69's `MemoObject` and
`MemoCallback` re-rendered every time; wrap those two values and they stop. That is the fix, and
it is why the two lessons are adjacent.

Note the direction of the dependency: **`useMemo` is usually pointless unless something
downstream compares references.** `memo` on a child, a dep array, or another `useMemo`.

### How expensive is "expensive"?

Guessing is the failure mode, so React gives you the method and the number:

> In general, unless you're creating or looping over thousands of objects, it's probably not
> expensive.

```js
console.time('filter array');
const visibleTodos = filterTodos(todos, tab);
console.timeEnd('filter array');
```

> If the logged time adds up to a significant amount (say, `1ms` or more), it might make sense to
> memoize that calculation.

That is a two-line experiment you can run in any component, and it is the entire justification
required before adding a `useMemo`. The example cell runs exactly this measurement on a real
calculation so you can see the shape of the numbers.

### Should you memoize everything?

> **No.** …There is no benefit to wrapping a calculation in `useMemo` in other cases. There is no
> significant harm to doing that either, so some teams choose to memoize as much as possible.
> However, code becomes less readable, and not all memoization is effective.

Two more caveats worth knowing. React treats the cache as a hint — it *"will throw away the
cached value"* in some situations, for example in development when you edit the component's
file, so behaviour must never depend on it. And every `useMemo` adds a dependency array you now
have to keep correct: a wrong one caches a stale value, which is a correctness bug bought with a
performance tool.

### The list that makes most of this unnecessary

React's own recommendation, and the best summary of this whole topic:

> 1. When a component wraps other components, let it accept JSX as children…
> 2. Prefer local state and don't lift state up further than necessary.
> 3. Keep your rendering logic pure.
> 4. Avoid unnecessary Effects that update state.
> 5. Try to remove unnecessary dependencies from your Effects.

You have been taught all five, in topics 22, 14 and 03. None of them is a performance trick;
they are just the correct way to write the code, and they happen to remove most of the reasons
you would need this lesson.

### Key Notes

- `useMemo` caches a value between renders; `useCallback` caches a function reference.
- Two valid reasons: the calculation is measurably slow, or something downstream compares
  identity (`memo`, a dep array).
- Measure with `console.time`; roughly 1 ms is the threshold worth acting on.
- A wrong dependency array turns a performance tool into a stale-value bug.

### Example

**Runnable — plain JS.** React's own measurement recipe, on a calculation big enough to matter.
Your numbers will differ from these — the point is the method and the ratio, not the value.

In [ ]:
// L70 — is this calculation actually expensive?

const l70Items = Array.from({ length: 20000 }, (_, i) => ({
  id: i,
  name: `item-${i}`,
  price: (i * 7919) % 1000,
  tag: ["a", "b", "c"][i % 3],
}));

function l70Visible(list, tag) {
  return list
    .filter((item) => item.tag === tag)
    .sort((a, b) => a.price - b.price || a.name.localeCompare(b.name))
    .slice(0, 20);
}

// the recipe, exactly as the docs give it
console.time("filter array");
const l70Rows = l70Visible(l70Items, "b");
console.timeEnd("filter array");

// the same calculation over a realistic list length
const l70Small = l70Items.slice(0, 200);
console.time("filter 200");
l70Visible(l70Small, "b");
console.timeEnd("filter 200");

// one measurement is noise: the first run pays for warm-up. average a few.
let l70Total = 0;
for (let i = 0; i < 20; i += 1) {
  const start = performance.now();
  l70Visible(l70Items, "b");
  l70Total += performance.now() - start;
}
console.log("20000 items, average of 20 runs (ms):", (l70Total / 20).toFixed(2));
console.log("top row:", l70Rows[0]);

// The verdict is a comparison against one number: is it 1ms or more?

### Exercise

Part 1 is **runnable — plain JS**; part 2 is **in the playground**.

1. Find the list length at which this calculation crosses 1 ms on **your** machine. Write
   `l70Threshold()` that measures `l70Visible` for lengths 200, 1000, 5000, 20000 and 50000
   (averaging a few runs each) and prints a table of length against milliseconds. Then say, in a
   comment, which of those lengths a typical screen would actually hold — and what that implies
   about how often `useMemo` is the right answer for filtering a list you are about to display.
2. **In the playground**, in `28-rerenders.jsx`: import `useMemo` and `useCallback`, wrap
   `config` in `useMemo(() => ({ label: "stable" }), [])` and `onPing` in
   `useCallback(() => {}, [])`, and press **App state** twice. Report the render counts for
   `MemoObject` and `MemoCallback` before and after. Then remove `memo` from `MemoObject` while
   keeping the `useMemo`, and say what the `useMemo` is now buying you.
3. In a comment: `useCallback(() => {}, [])` around a handler passed to a plain `<button>`. Is
   that worth anything? Answer in terms of who compares the reference.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** The stale-dependency bug, which is the real cost of memoizing casually.

Write `l70Cache(compute, deps)` — a tiny stand-in for `useMemo` that stores the last deps and
last result and recomputes only when a dep fails `Object.is`. Then use it to cache
`l70Visible(items, tag)` while **forgetting `tag` in the dependency list**, and call it twice
with different tags.

Print both results. Then answer in comments: what does the second call return, is that a
performance problem or a correctness problem, and which of the two failures — a missing
dependency or a missing `useMemo` — would you rather ship?

In [ ]:
// Your code here

## LESSON 71 — React Compiler

The last two lessons taught you to place `memo`, `useMemo` and `useCallback` by hand, after
measuring. React Compiler is the argument that you should mostly not have to.

### What it is

> React Compiler is a new build-time tool that automatically optimizes your React app. It works
> with plain JavaScript, and understands the Rules of React, so you don't need to rewrite any
> code to use it.

Build-time is the key word. It is not a runtime library and not an API you call — it reads your
components during the build and emits versions with the memoization already inserted:

> React Compiler automatically applies the optimal memoization, ensuring your app only re-renders
> when necessary.

It does the two things you did by hand in lessons 69 and 70: skipping cascading re-renders of
child components, and skipping expensive calculations during render. React's own framing of why:

> This manual memoization is tedious, easy to get wrong, and adds extra code to maintain.

Every word of that is something you experienced in the last two lessons — a dependency array you
have to keep correct, a `memo` that silently does nothing because of one object literal.

### Its status, and what it needs from you

> React Compiler is now stable and has been tested extensively in production

It is still opt-in — a build-tool plugin you add — though React notes that *"in the future some
features may require the compiler in order to fully work."*

The one condition matters more than the install: it *"understands the Rules of React"*, and how
well it can optimise a codebase *"will depend on the health of your codebase and how well you've
followed the Rules of React"*. Those are the rules this course has been teaching all along —
components pure during render (topic 03), never mutating state (L27), never writing a ref during
render (L51), and the Rules of Hooks (L37). Code that breaks them is code the compiler will
either skip or, worse, was already subtly broken.

Adoption is a build change, not a code change. On Vite it is a dev dependency and a plugin entry,
plus the ESLint rules that flag components the compiler cannot optimise. **This course does not
install it in the playground**, because it would change how every other experiment builds for the
sake of one lesson — and because the point of the lesson is what it does, not how to configure it.

### What it does not do

Two limits are worth carrying:

- It memoizes **React components and hooks, not every function**. A genuinely expensive plain
  function is still your problem: React's docs say you *"may want to consider implementing its own
  memoization outside of React"*.
- Memoization *"is not shared across multiple components or hooks"* — the same expensive
  calculation in two components runs twice.

And the one this whole topic exists to protect: **the compiler does not remove the need to
measure.** It removes hand-written memoization. A keystroke that reaches four hundred components
because state lives in `App` still reaches four hundred components — some of them now skipped,
but the design problem from LESSON 66 is untouched. Measure first, still.

### What this means for what you write

Nothing changes in this course. You will not use the compiler in the mini-projects, and every
lesson after this one is written as if it does not exist — which is also how you should treat it
until a project measurably needs it.

What it *should* change is your instinct about manual memoization. Given a choice between
scattering `useMemo` through a codebase and turning on a compiler that does it more reliably,
the second is now the better default for a new project. Write correct, rule-following components;
add memoization by hand only where you measured a problem the compiler did not solve.

### Key Notes

- A build-time tool, now stable, that inserts memoization for you. Not an API you call.
- It depends on your code following the Rules of React — the ones this course has taught.
- It memoizes components and Hooks, not arbitrary functions, and does not share caches.
- It does not replace measuring, and it does not fix state that lives in the wrong place.

### Example

**No cell, and no playground.** There is nothing to run: the compiler's whole output is a build
artefact, and installing it here would change every other experiment in the shared playground for
one lesson's sake.

If you want to see it, the honest way is a scratch project of your own — `react-scratch` from
LESSON 2 — following the current instructions at
<https://react.dev/learn/react-compiler/installation>. Read those rather than any snippet
reproduced here; the plugin's configuration has changed more than once and a version copied into
a notebook goes stale silently.

### Exercise

**No code.** Four questions. Write the answers down — this is a lesson about judgement, and the
answers are short.

1. Your app has one screen that stutters when typing. You have not opened the Profiler. Does
   turning on React Compiler solve it? Answer in terms of what the compiler changes and what it
   does not.
2. A colleague says "we don't need to understand `memo` any more". Give the two-part answer: what
   they are right about, and why lessons 68–70 are still worth having.
3. Your component mutates an array in state directly, and everything currently works. What does
   the compiler's requirement — that you follow the Rules of React — mean for that component?
4. You have a plain function that scores 6000 records and is called from two components. Which
   part of that does the compiler handle, and which part is still yours?

### Mini challenge

**Runnable — plain JS.** One small thing you can genuinely measure without the compiler: the
memoization it *cannot* do for you.

Write `l71Score(records)` that does real work — sum, average and standard deviation over the
array — and measure it with `console.time` for 6000 records. Now call it from two different
"components" (two plain functions) in the same "render", and measure the total.

Then write `l71Memoized`, a version that caches by the array reference (a `WeakMap` or a stored
last-argument), call it from both, and measure again.

Answer in comments: how much did the second call cost with and without your cache? React's docs
say memoization *"is not shared across multiple components or hooks"* — which of your two
versions has that limitation, and which one just solved it? And why is a plain module-level
cache, rather than a Hook, the right shape for this particular problem?

In [ ]:
// Your code here

## LESSON 72 — `useTransition`

Everything so far in this topic made renders cheaper or fewer. This lesson is different: it
changes **when** a render is allowed to block the user.

> `useTransition` is a React Hook that lets you render a part of the UI in the background.

```jsx
const [isPending, startTransition] = useTransition();
```

Two things come back:

> 1. The `isPending` flag that tells you whether there is a pending Transition.
> 2. The `startTransition` function that lets you mark updates as a Transition.

### Urgent and non-urgent, in one handler

A search box is the standard case, and it contains both kinds of update at once:

```jsx
function handleChange(event) {
  const next = event.target.value;

  setText(next);                                // URGENT — the input must show the keystroke

  startTransition(() => {
    setQuery(next);                             // non-urgent — the expensive list may lag
  });
}
```

Read the two lines carefully, because the whole lesson is in the difference between them.

**The input's own state is never in a Transition.** React is explicit that you *cannot* use
Transitions for state that controls a text input — controlled input state must update
synchronously, or the field stops keeping up with the keys. If typing feels broken, this is the
mistake.

What goes in the Transition is the **expensive derived work**: the filtered list, the big table,
the chart that redraws.

### What it actually buys — measured

Playground experiment 29 has both modes behind a checkbox, so you can run the same eight
keystrokes each way. Typing `person-1` one character at a time, and measuring only the time
React blocked the typing loop:

| mode | total blocking for 8 keystrokes |
|---|---|
| plain `setQuery` | **≈ 270 ms** (about 34 ms a keystroke) |
| inside `startTransition` | **≈ 6 ms** |

Both modes end with the same list on screen. The difference is that the second one never made
the browser wait for it. React's own description:

> Transitions do not block the user from interacting with the page.

And the reason it can do that:

> A state update marked as a Transition will be interrupted by other state updates.

The list render started for `perso` is abandoned the moment `person` arrives. Nothing is queued
up behind your typing.

### The part nobody mentions

A Transition alone is not enough, and the experiment proves it. With the expensive list **not**
wrapped in `memo`, running the same eight keystrokes produced:

| | renders of the expensive list |
|---|---|
| plain `setQuery` | 8 |
| inside `startTransition` | **16** |

Twice as many — the opposite of what you were hoping for. LESSON 68 explains it: the urgent
`setText` re-renders the component, and the expensive child re-renders **with the old query**
simply because its parent rendered; then the Transition renders it again with the new one.

Adding `memo` to the list brings both modes back to 8, because during the urgent render the
child's `query` prop has not changed and React skips it. So:

> A Transition moves expensive work off the urgent path. `memo` is what stops the urgent path
> from doing that work anyway.

This is why this lesson comes after 68–70 and not before.

### Pending UI, and when not to bother

`isPending` is true while the non-urgent render is outstanding — enough for a subdued list, a
small spinner, or nothing at all. Show something calm: a full "Loading…" that replaces the old
results is worse than the slightly stale results, and losing the old content is exactly what
Transitions exist to avoid.

Two more caveats from the docs, both easy to trip over: the function you pass *"is called
immediately"*, so state updates inside a `setTimeout` are not part of the Transition; and after
an `await`, updates need their own `startTransition` to stay in one.

And the honest limit: if the non-urgent render takes 5 ms, a Transition changes nothing you can
perceive. Measure first — the same rule as the rest of this topic.

### Key Notes

- `const [isPending, startTransition] = useTransition()`; mark the **expensive derived update**,
  never the controlled input's own state.
- Measured: ≈ 270 ms of blocking over eight keystrokes becomes ≈ 6 ms.
- Transitions are interruptible — a newer keystroke abandons the in-flight render.
- Without `memo` on the expensive child, a Transition can *double* its renders. Both tools, or
  neither.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/29-transitions.jsx`.

The checkbox switches between `setQuery(next)` and `startTransition(() => setQuery(next))` with
nothing else changed. Type quickly with it off, then on. Watch three things: the input's own
responsiveness, the `pending` label, and the fact that the list is briefly one keystroke behind —
which is the point, not a bug.

`window.__resultsRenders` holds the render count for the expensive list, so you can reproduce the
tables above yourself.

### Exercise

**In the playground**, in `29-transitions.jsx`.

1. Reproduce the blocking measurement. Type `person-1` one character at a time in each mode and
   record how long the keystrokes blocked. You can measure by hand — type fast and watch the
   input — or with the Profiler from LESSON 35.
2. Remove `memo` from `Results` and repeat the render count in both modes using
   `window.__resultsRenders`. Explain the number you get for the Transition mode using LESSON 68's
   two reasons for a re-render.
3. Now do the thing the lesson tells you not to: put `setText(next)` inside the
   `startTransition` callback as well, and type a sentence quickly. Describe exactly what breaks,
   and connect it to React's rule about controlled inputs.
4. Reduce `ROWS` to 200 items and repeat part 1. Is the Transition still worth having? Answer in
   terms of what you measured, not what the API is for.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** Transitions and debouncing (LESSON 42) are often confused, and the
difference is easiest to see as a timeline.

Write `l72Timeline(keystrokes, workMs, strategy)` that simulates typing — one keystroke every
50 ms — and returns, for each strategy, how many times the expensive work would run and at what
simulated times:

- `"none"` — the work runs on every keystroke
- `"debounce"` — the work runs 400 ms after the last keystroke
- `"transition"` — the work starts on every keystroke but any run still in progress is
  abandoned when the next keystroke arrives

Run it for eight keystrokes with `workMs = 120`. Then answer in comments: which strategy shows
the user something soonest, which does the least total work, and why the honest answer for a
search box that hits the network is usually **both** — debounce the request, and mark the
rendering of the results as a Transition.

In [ ]:
// Your code here

## LESSON 73 — Long lists, and loading code on demand

Two separate techniques, both about the same thing: not doing work the user has not asked for.

### Long lists

Rendering a list is rendering every item. Experiment 30 measures it: creating the elements for
50 rows takes well under a millisecond, and the same code for 5000 rows takes about **48 ms** —
and leaves 5000 nodes in the DOM, which the browser then has to lay out on every change.

The fixes, in the order you should try them:

1. **Don't render 5000 rows.** Paginate, or filter, or show the top N with a "show more". Almost
   every list that is slow is a list nobody wanted to see all of.
2. **Keep keys stable.** LESSON 20's rule matters more as lists grow: with `key={item.id}`, React
   moves existing DOM nodes when the order changes; with `key={index}` it rewrites the contents of
   every row after the change point. Same output, much more work — and any state inside those
   rows lands on the wrong item.
3. **Windowing**, if you genuinely must show tens of thousands of rows: render only the visible
   slice and translate it as the user scrolls. This is what libraries like `react-window` do, and
   it is out of scope here — the point is to know the word before you need it.

`memo` on a row component helps only when the rows' props are stable — which, after LESSON 69,
you know means the parent must not rebuild each row's object during render.

### `lazy` + `Suspense`: loading code on demand

Everything you import is in the bundle the user downloads before anything appears. A chart
library used on one screen out of nine is still in that first download — unless you split it out.

```jsx
import { Suspense, lazy } from "react";

const Chart = lazy(() => import("./Chart.jsx"));

function Report({ show }) {
  return (
    <div>
      {show && (
        <Suspense fallback={<p>Loading the chart…</p>}>
          <Chart />
        </Suspense>
      )}
    </div>
  );
}
```

Three parts:

- `import("./Chart.jsx")` — a **dynamic import**, which returns a Promise for the module. The
  build tool sees it and puts that module in a separate file.
- `lazy(...)` wraps that into a component you can render normally.
- `<Suspense fallback={…}>` says what to show while the code is arriving. Without it, React has
  nothing to render and throws.

Experiment 30's production build shows the split as two files:

```text
dist/assets/30-lazy-child-Cq1Xl1Xe.js    0.42 kB
dist/assets/index-bAf_c7IG.js          221.90 kB
```

And in the browser: **no request for the child module until the button is pressed**, then the
fallback appears, then the component. Measured, in that order.

### Where to split, and where not to

Splitting has a cost — a network round trip at the moment the user asks for something. Good
places: a route the user may never visit (topic 20's routes are the classic split point), a heavy
editor or chart behind a button, an admin screen. Bad places: a component that appears on first
paint (you have added a round trip to the critical path for nothing), or a 2 kB component (the
request costs more than the code).

> **Suspense here is only for code.** `lazy` is the one Suspense use you have so far. Ordinary
> data loading still uses the explicit loading / error / empty / success states from LESSON 46 —
> Suspense is not a replacement for that. LESSON 77 introduces the second legitimate use, `use`.

### Key Notes

- The cost of a list is per item: 50 rows ≈ 0 ms, 5000 rows ≈ 48 ms and 5000 DOM nodes.
- Stable keys are a performance rule as well as a correctness one; index keys rewrite rows.
- `lazy(() => import(...))` + `<Suspense fallback>` moves code out of the first download.
- Split what the user may never open. Never split what is on screen at first paint.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/30-lazy-suspense.jsx`.

Open the **Network** tab and reload before pressing anything. `30-lazy-child` is not in the list.
Press **load the chart** and it appears — that request is the split, and the fallback is what the
user sees while it happens.

Then press **show all 5000 rows** and watch `window.__lastListMs`, which holds how long creating
the row elements took.

### Exercise

**In the playground**, in `30-lazy-suspense.jsx`.

1. Record `window.__lastListMs` for 50 rows and for 5000 rows. Then change the row list to use
   `key={index}` instead of `key={row.id}`, add a button that removes the first row, and compare
   what the browser has to do in each case. (DevTools' *Paint flashing* or the Elements panel
   makes this visible.)
2. Throttle the network to **Slow 3G** in DevTools and press **load the chart** again. Now that
   the fallback is on screen for a real length of time, does its wording still work? Rewrite it if
   not.
3. Move `<Suspense>` **outside** the `{showChart && …}` so it is always mounted, and check the
   behaviour is unchanged. Then remove `<Suspense>` entirely and describe the error React gives
   you.
4. Add a second lazy component that is rendered on first paint, and explain — in one sentence —
   why that is a mistake even though it works.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** A splitting budget, from real numbers.

Your bundle is 220 kB, and four features contribute to it: a chart library (95 kB, used on one
screen of nine), a date picker (30 kB, used in two forms), a markdown editor (60 kB, opened by a
button roughly one session in twenty), and your own code (35 kB, everywhere).

Write `l73Budget(features)` that returns, for each feature, the first-download size with and
without splitting it, and the total saved. Then decide each case with a rule you can state, and
print your decision alongside each number.

Finally, in a comment: one of these four should **not** be split even though it is large and the
arithmetic says otherwise. Which, and what does the arithmetic leave out?

In [ ]:
// Your code here

> **Topic 23 complete — LESSON 68 to 73.** Read the sequence back: two reasons for a re-render →
> `memo` and identity → `useMemo`/`useCallback` and a measurement → a compiler that does most of
> it for you → Transitions for work that must not block typing → and not shipping work at all.
>
> The rule the topic opened with is the one to keep: **make it correct, measure, then optimise.**
> Every tool here has a measurement that justifies it, and every one of them is dead weight
> without it.